In [0]:
############################################## BRONZE LAYER ###############################################################
# Read the csv database
catalog = "workspace"
schema = "default" 
volume = "my_volume"
file_name = "netflix_titles.csv"
bronze_table_name = "bronze_netflix_titles"
path_volume = "/Volumes/" + catalog + "/" + schema + "/" + volume
from pyspark.sql.types import *
from pyspark.sql.functions import *
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .csv(f"{path_volume}/{file_name}")
)

# Cleaning up at ingestion: susbtituting any \n character with a space, in any col that may contain text
text_cols = ["title", "description", "cast", "director"]
for c in text_cols:
    df = df.withColumn(c, regexp_replace(col(c), r"\n", " "))

df.filter(col("show_id") == "s8420").show(truncate=False)
# checking that the numbers of rows in the table equals to 8807, ie the official number
df.count()
# creating a DeltaTable
df.write.mode("overwrite").saveAsTable(bronze_table_name)

In [0]:
############################################# SILVER LAYER #######################################################
df = spark.table(bronze_table_name)
# drop duplicates
df = df.dropDuplicates()
# rename the column listed_in -> genres
df = df.withColumnRenamed("listed_in", "genres")
display(df)

In [0]:
# convert date_added from string to date format
df = df.withColumn(
    "parsed_date",
    to_date(trim(col("date_added")), "MMMM d, yyyy")
)
display(df)

In [0]:
# trim whitespaces in each col
for c in df.columns:
  df = df.withColumn(c, trim(col(c)))

# format release_year col from string to int
df = df.withColumn("release_year", col("release_year").cast("int"))
# format duration col from int to string
df = df.withColumn("duration", col("duration").cast("string"))
# format genre col in a standardized manner
df = df.withColumn("genres", regexp_replace(col("genres"), r"\s*, *\s", ","))
# Handle NULL values
df = df.fillna(
    {
        "director": "unknown",
        "cast": "unknown",
        "country": "unknown",
        "date_added": "unknown",
        "rating": "unknown",
        "genres": "unknown",
        "description": "unknown"
    }
) 

# Saving the silver table to DataFrame
df.write.mode("overwrite").saveAsTable("silver_netflix_titles")

In [0]:
####################### GOLDEN LAYER - DATA ANALYSIS ################################################## 
df = spark.table("silver_netflix_titles")
df.printSchema()
# QUESTION 1: which is the movies/shows ratio?
df.groupBy("type").count().display()

In [0]:
# Question 2: 10 most common Netflix genres? 
genres_df =  df.withColumn("genre", explode(split(col("genres"), ",")))
genres_df.groupBy("genre").count().orderBy(col("count").desc()).limit(10).display()  

In [0]:
# Question 3: Netflix 10 top countries (for content production)?
countries_df = df.withColumn("country", explode(split(col("country"), ",")))
countries_df = countries_df.withColumn("country", trim(col("country")))
countries_df = countries_df.filter(col("country") != "unknown")

top_countries_df = countries_df.groupBy("country").count().orderBy("count", ascending=False).limit(10).display()
#df.groupBy("country").count().orderBy("count", ascending=False).limit(10).display()

In [0]:
# Question 4: longest movie and longest TV show?
movies = df.filter(col("type")=="Movie")
movies = movies.withColumn("duration_minutes", regexp_extract(col("duration"), r"(\d+)", 1).cast("int"))
movies.groupBy("type").agg(max("duration_minutes")).display()

tv_shows = df.filter(col("type")=="TV Show")
tv_shows = tv_shows.withColumn("duration", regexp_extract(col("duration"), r"(\d+)", 1).cast("int"))
tv_shows.groupBy("type").agg(max("duration")).display()
#df.groupBy("type").agg(max("duration")).display()

In [0]:
# Question 5: content added over the years?
df.withColumn("year_added", year(col("parsed_date"))) \
  .groupBy("year_added", "type") \
  .count() \
  .orderBy("year_added").display()